# Spotify Data Analysis – DE Milestone 1 (CIE)

**Objective:** Read a CSV dataset using Pandas, understand and transform the data, identify and handle missing values using mean imputation wherever applicable, generate insights, and visualize the findings.

**Dataset:** `spotify_data clean.csv`  
**Records:** 8,582 | **Columns:** 15

> Keep the CSV file in the same folder as this notebook before running the cells.

## 1. Import required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

print("Libraries imported successfully!")

## 2. Read the CSV dataset

In [ ]:
df = pd.read_csv("spotify_data clean.csv")

print("Dataset loaded successfully!")
print("Shape of dataset:", df.shape)
df.head()

## 3. Understand the dataset

In [ ]:
# Display first and last records
display(df.head())
display(df.tail())

# Dataset structure and data types
df.info()

# Statistical summary of numerical columns
display(df.describe())

# Column names
print("Columns:", df.columns.tolist())

### 3.1 Check duplicate records

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate track IDs:", df["track_id"].duplicated().sum())

## 4. Data transformation

We will:
- Convert `album_release_date` into a datetime column.
- Create a `release_year` column.
- Create a `duration_category` column.
- Create a `popularity_category` column.
- Remove identifier columns that are not useful for our summary analysis (`track_id` and `album_id`).

The original dataset remains unchanged in `df`; transformations are performed on `spotify`.

In [ ]:
spotify = df.copy()

# Convert release date from text to datetime
spotify["album_release_date"] = pd.to_datetime(
    spotify["album_release_date"], errors="coerce"
)

# Extract release year
spotify["release_year"] = spotify["album_release_date"].dt.year

# Categorize track duration
spotify["duration_category"] = np.where(
    spotify["track_duration_min"] < 3, "Short",
    np.where(spotify["track_duration_min"] <= 5, "Medium", "Long")
)

# Categorize track popularity (Spotify popularity scale: 0–100)
spotify["popularity_category"] = pd.cut(
    spotify["track_popularity"],
    bins=[-1, 40, 70, 100],
    labels=["Low", "Medium", "High"]
)

# Drop unique ID columns because they do not help with these analyses
spotify.drop(columns=["track_id", "album_id"], inplace=True)

display(spotify.head())
print("Updated shape:", spotify.shape)

## 5. Identify and handle missing values

First, we count missing values in every column. Mean imputation is suitable for numerical columns, not text/category columns. In this dataset, the numerical columns currently have no missing values, so there are no numerical values to replace with a mean. The code below still performs mean imputation wherever numerical missing values exist. For text columns, we use `Unknown` because a mean cannot be calculated for text.

In [ ]:
# Count null values before handling
null_counts = spotify.isnull().sum()
print("Missing values before handling:")
display(null_counts[null_counts > 0])

# Mean imputation for numerical columns, wherever applicable
numeric_cols = spotify.select_dtypes(include=np.number).columns

for col in numeric_cols:
    if spotify[col].isnull().any():
        mean_value = spotify[col].mean()
        spotify[col] = spotify[col].fillna(mean_value)
        print(f"{col}: missing values replaced with mean = {mean_value:.2f}")

# Fill missing categorical/text values with 'Unknown'
categorical_cols = spotify.select_dtypes(include=["object", "category"]).columns

for col in categorical_cols:
    if spotify[col].isnull().any():
        spotify[col] = spotify[col].fillna("Unknown")

# Check remaining null values
print("Missing values after handling:")
display(spotify.isnull().sum())

### 5.1 Interpretation of missing values

In the uploaded file:
- `artist_name` has 3 missing values.
- `artist_genres` has 3,361 missing values.
- All numerical columns have 0 missing values.

Therefore, mean imputation is not required for the current numerical columns. We fill missing artist names and genres with `Unknown`, because these are categorical/text values and calculating their mean is not meaningful.

## 6. Insight generation

### 6.1 Top 10 artists by number of tracks

In [ ]:
top_artists = spotify["artist_name"].value_counts().head(10)
display(top_artists.rename("number_of_tracks").to_frame())

### 6.2 Top 10 tracks by popularity

In [ ]:
top_tracks = spotify.nlargest(10, "track_popularity")[
    ["track_name", "artist_name", "track_popularity"]
]
display(top_tracks)

### 6.3 Average popularity by album type

In [ ]:
album_type_analysis = (
    spotify.groupby("album_type")["track_popularity"]
    .agg(["count", "mean", "max"])
    .sort_values("mean", ascending=False)
)
display(album_type_analysis.round(2))

### 6.4 Explicit vs non-explicit tracks

In [ ]:
explicit_analysis = (
    spotify.groupby("explicit")["track_popularity"]
    .agg(["count", "mean"])
)
display(explicit_analysis.round(2))

### 6.5 Number of tracks released each year

In [ ]:
yearly_tracks = (
    spotify.groupby("release_year")
    .size()
    .sort_index()
)
display(yearly_tracks.tail(15).rename("number_of_tracks").to_frame())

### 6.6 Average track duration by album type

In [ ]:
duration_analysis = (
    spotify.groupby("album_type")["track_duration_min"]
    .mean()
    .sort_values(ascending=False)
)
display(duration_analysis.round(2).rename("average_duration_minutes").to_frame())

## 7. Data visualization

### 7.1 Top 10 artists by track count – Bar chart

In [ ]:
plt.figure(figsize=(11, 5))
top_artists.sort_values().plot(kind="barh")
plt.title("Top 10 Artists by Number of Tracks")
plt.xlabel("Number of Tracks")
plt.ylabel("Artist")
plt.tight_layout()
plt.show()

### 7.2 Distribution of track popularity – Histogram

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(spotify["track_popularity"], bins=20, edgecolor="black")
plt.title("Distribution of Track Popularity")
plt.xlabel("Track Popularity")
plt.ylabel("Number of Tracks")
plt.tight_layout()
plt.show()

### 7.3 Average popularity by album type – Bar chart

In [ ]:
plt.figure(figsize=(8, 5))
album_type_analysis["mean"].plot(kind="bar")
plt.title("Average Track Popularity by Album Type")
plt.xlabel("Album Type")
plt.ylabel("Average Track Popularity")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 7.4 Explicit vs non-explicit tracks – Bar chart

In [ ]:
plt.figure(figsize=(7, 5))
explicit_analysis["mean"].plot(kind="bar")
plt.title("Average Popularity: Explicit vs Non-explicit Tracks")
plt.xlabel("Explicit Track (False = No, True = Yes)")
plt.ylabel("Average Track Popularity")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 7.5 Tracks released per year – Line chart

In [ ]:
yearly_tracks.plot(figsize=(11, 5), kind="line", marker="o")
plt.title("Number of Tracks by Album Release Year")
plt.xlabel("Release Year")
plt.ylabel("Number of Tracks")
plt.tight_layout()
plt.show()

### 7.6 Track duration distribution – Histogram

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(spotify["track_duration_min"], bins=25, edgecolor="black")
plt.title("Distribution of Track Duration")
plt.xlabel("Duration (minutes)")
plt.ylabel("Number of Tracks")
plt.tight_layout()
plt.show()

## 8. Key findings and conclusion

Use the outputs generated above to discuss your findings. The initial inspection of this dataset shows:

1. The dataset contains 8,582 tracks and 15 original columns.
2. Taylor Swift has the highest number of tracks in this dataset (324).
3. Album records have a higher average track popularity (about 55.66) than singles (about 46.36) and compilations (about 40.51).
4. Explicit tracks have a higher average popularity (about 57.82) than non-explicit tracks (about 50.53) in this dataset. This is an association, not proof that explicit content causes popularity.
5. The dataset has 3 missing artist names and 3,361 missing genre values.
6. No numerical columns have missing values, so numerical mean imputation is not needed for this particular CSV.
7. The release-year chart can be used to identify years with more records in the dataset.

**Conclusion:** This assignment demonstrates CSV reading, dataset exploration, data type conversion, feature creation, dropping unnecessary columns, null-value handling, aggregation, insight generation, and visualization using Python libraries.

## 9. Save the cleaned dataset (optional)

In [ ]:
spotify.to_csv("spotify_cleaned.csv", index=False)
print("Cleaned dataset saved as spotify_cleaned.csv")